# Marine 5-Day Forecast — PyTorch: 12-Model Comparison (LSTM, XGBoost, N-BEATS, N-HiTS, DLinear, TiDE, TSMixer, Harmonic-Residual, iTransformer, PatchTST, DeepAR, Gaussian Process)

This notebook is a **PyTorch port** of `Marine_Forecast_LSTM_XGBoost.ipynb`, extended with
9 additional forecasters for comparison — covering recurrent, gradient-boosted, MLP/linear,
domain-specific, and attention-based architecture families:

1. **LSTM** (PyTorch, replacing the original TensorFlow/Keras model) — recursive rollout.
2. **XGBoost** — unchanged, direct multi-horizon, one model per parameter (kept as the
   strong classical baseline).
3. **N-BEATS** (pure PyTorch implementation) — direct multi-horizon, doubly-residual
   stack of fully-connected blocks. No recursion, so no error compounding over 120 steps.
4. **N-HiTS** (pure PyTorch implementation) — like N-BEATS but each stack first
   **pools the input at a different rate** (24h / 12h / 1h, matching diurnal, semidiurnal
   tidal, and fine-grained scales in this data) then **hierarchically interpolates** a
   coarse basis up to the full horizon. Designed specifically for **long-horizon**
   forecasting, which is exactly this problem (120-step / 5-day horizon).
5. **DLinear** (Zeng et al., AAAI 2023) — a single linear layer over trend/seasonal
   decomposition. The field's standard sanity-check baseline.
6. **TiDE** (Das et al., Google, TMLR 2023) — dense (MLP-only) encoder-decoder that
   explicitly consumes known future covariates (our calendar features) plus a global
   linear residual connection.
7. **TSMixer** (Chen et al., Google, 2023) — all-MLP architecture alternating
   time-mixing and feature-mixing blocks, explicitly modeling cross-parameter coupling.
8. **Harmonic Regression + Residual MLP Hybrid** — a domain-specific approach matching
   the literature review's #1 recommendation for **tidal level**
   (`MARINE_FORECASTING_IMPLEMENTATION_GUIDE.md`, §1.1): fit deterministic tidal/diurnal
   harmonics (M2, S2/synoptic, K1, O1 + diurnal cycle) by least squares, then train a
   small MLP on the *residual* only.
9. **iTransformer** (Liu et al., 2023) — "inverts" the standard Transformer: each
   *variate* becomes a token, so self-attention learns cross-parameter dependencies
   directly. Fills the one architectural gap left by models 1-8: no attention.
10. **PatchTST** (Nie et al., ICLR 2023) — splits each channel's history into patches and
    runs a channel-independent Transformer (shared weights across all 21 channels) over
    the patch sequence. One of the most consistently cited SOTA results in recent
    multivariate-forecasting literature.
11. **DeepAR** (Salinas et al., Amazon) — a probabilistic RNN: instead of predicting a
    single number, it predicts a Gaussian (mean, std) at every step, trained by
    maximizing likelihood rather than minimizing MSE. Forecasting 120 hours runs **100
    Monte-Carlo sample paths** through the recursive rollout; the point forecast is
    their mean, and — uniquely among all 12 models — their *spread* gives a genuine,
    horizon-growing uncertainty band.
12. **Gaussian Process** (via GPyTorch) — a Bayesian, per-parameter model with an
    additive kernel: two **periodic kernels frozen at the known true periods** (24h
    diurnal, 12.42h M2 tide) plus a local RBF kernel for everything else. Gives a
    calibrated 95% credible interval natively, with no sampling required.

Models 9-10 were added after reviewing `latest_research.txt`, a 2023-2026 survey of
multivariate time-series forecasting research, specifically to cover the attention-based
model family (Crossformer / iTransformer / PatchTST / Pathformer) that survey identifies
as a leading SOTA direction, complementing the MLP/linear family already covered by
DLinear/TSMixer/TiDE.

Models 11-12 were added after reviewing `ML models_few_more.txt`, a survey covering
foundation models (Chronos, Moirai, Lag-Llama), weather/climate-specific architectures
(GraphCast, Pangu-Weather, FourCastNet), Gaussian Processes, Neural ODEs, PINNs, and
spatio-temporal GNNs. **Most of that survey doesn't fit this problem and was deliberately
left out**:
- GraphCast / Pangu-Weather / FourCastNet / Neural Operators (FNO) all require a spatial
  **grid or mesh** (they forecast whole weather fields over a map) — this dataset is a
  single buoy location with no spatial dimension, so these architectures have nothing to
  operate on here.
- Chronos / Moirai / Lag-Llama are pretrained foundation models meant for zero-shot
  transfer from massive external corpora — using them well means downloading large
  pretrained checkpoints, not training from scratch on 70 days of one site's data; out of
  scope for this notebook's from-scratch, fully-reproducible approach.
- PINNs and Neural ODEs need either known governing PDEs/ODEs or continuous/irregular
  sampling to earn their complexity; this data is regularly-sampled hourly and we already
  have a domain-physics model (Harmonic-Residual) for the one parameter (tides) with a
  clean known equation.
- Spatio-temporal GNNs need a graph; the one place a 'graph' could mean something here is
  *across parameters*, which iTransformer and TSMixer already cover.

What **does** fit a single-location multivariate tabular problem from that survey:
**DeepAR** (probabilistic RNN forecasting) and **Gaussian Processes** (GPyTorch) — both
explicitly recommended in that survey for exactly this kind of comparatively small,
single-site, multivariate setting, and both add something genuinely new: **uncertainty
quantification**, which none of models 1-10 provide.

All twelve models forecast all 16 marine parameters **5 days (120 hours) ahead**, validated
against the same held-out ground truth, so results are directly comparable. The notebook
also ranks every parameter by its best model (§18) and tests whether any 2-model
ensemble beats the single best model (§19) — built for ship mooring / docking decision
support.

Environment: conda env `marinepred` (PyTorch + XGBoost, no TensorFlow needed).

## 0. Setup

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__, "| device:", device, "| XGBoost:", xgb.__version__)


## 1. Load data

Reuses `marine_data_75days.csv` generated by the companion TensorFlow notebook (same
75-day synthetic dataset: 70 days history + 5 days held out), so the two notebooks are
directly comparable. If the file is missing, re-run `generate_marine_data.py` first.

In [ ]:
DATA_PATH = "marine_data_75days.csv"
HORIZON   = 120   # forecast 5 days = 120 hours
LOOKBACK  = 72    # input window = 3 days

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"]).set_index("timestamp")
print(f"Loaded: {df.shape[0]} hourly rows x {df.shape[1]} parameters")
print(f"Range : {df.index.min()} -> {df.index.max()}")
df.head()


## 2. Prepare data for modeling

Same preprocessing as the TensorFlow notebook:
- Circular `sin`/`cos` encoding of wind direction.
- Calendar features (`hour`, `day-of-year` as sin/cos) — known in advance, so true future
  values are fed to the models rather than forecast.
- Train/test split: last 120 hours (5 days) held out. Scaling uses **train statistics only**.

In [ ]:
wd_rad = np.deg2rad(df["wind_direction_deg"])
df["wind_dir_sin"] = np.sin(wd_rad)
df["wind_dir_cos"] = np.cos(wd_rad)

target_cols = [
    "significant_wave_height_m", "wave_period_s", "wind_speed_ms",
    "wind_dir_sin", "wind_dir_cos", "tidal_level_m", "current_speed_ms",
    "sea_surface_temp_c", "salinity_psu", "conductivity_mscm",
    "air_pressure_hpa", "air_temp_c", "relative_humidity_pct",
    "dew_point_c", "precipitation_mmh", "solar_radiation_wm2", "visibility_km",
]
model_data = df[target_cols].copy()

idx = model_data.index
model_data["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
model_data["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
model_data["doy_sin"]  = np.sin(2 * np.pi * idx.dayofyear / 365)
model_data["doy_cos"]  = np.cos(2 * np.pi * idx.dayofyear / 365)

feature_cols  = list(model_data.columns)
calendar_cols = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]
n_targets, n_features = len(target_cols), len(feature_cols)
target_idx   = [feature_cols.index(c) for c in target_cols]
calendar_idx = [feature_cols.index(c) for c in calendar_cols]

train_df = model_data.iloc[:-HORIZON].copy()
test_df  = model_data.iloc[-HORIZON:].copy()

# standardize using TRAIN stats only (no leakage)
mean = train_df.mean()
std  = train_df.std().replace(0, 1)
train_scaled = (train_df - mean) / std
full_scaled  = (model_data - mean) / std

future_calendar_scaled = full_scaled[calendar_cols].iloc[-HORIZON:].values.astype(np.float32)

print(f"Train: {train_df.shape[0]} hours ({train_df.shape[0]//24} days)")
print(f"Test : {test_df.shape[0]} hours ({test_df.shape[0]//24} days)  <- held-out validation")


## 3. Shared training utilities

A single training loop (Adam + `ReduceLROnPlateau` + early stopping) is reused for the
LSTM, N-BEATS and N-HiTS PyTorch models below, so training behavior is consistent and
comparisons aren't confounded by different optimization schedules.

In [ ]:
def train_model(model, X_tr, Y_tr, X_val, Y_val, epochs=200, batch_size=64,
                lr=1e-3, patience=20, name=""):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=6)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait = float("inf"), None, 0
    n = len(X_tr)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            xb, yb = X_tr[b].to(device), Y_tr[b].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val.to(device)), Y_val.to(device)).item()
        sched.step(val_loss)
        if val_loss < best_val - 1e-5:
            best_val, wait = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    print(f"{name:10s} best_val_loss={best_val:.4f}  epochs_run={ep+1:3d}  time={time.time()-t0:5.1f}s")
    return model


## 4. Model 1 — PyTorch LSTM (recursive rollout)

Direct port of the TensorFlow model: 2-layer LSTM (96 → 48 units) + dropout, trained as
multivariate **sequence-to-one** (given the last 72 hours of all features, predict the
next hour). To get a 5-day forecast we **roll it forward recursively** 120 times,
overwriting the calendar channels with their true future values at each step (the clock
is known in advance).

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden1=96, hidden2=48, dropout=0.2):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, hidden1, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.drop2 = nn.Dropout(dropout)
        self.head  = nn.Linear(hidden2, n_features)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.drop1(x)
        _, (h, _) = self.lstm2(x)
        return self.head(self.drop2(h[-1]))


def make_seq2one(arr, lookback):
    X, y = [], []
    for i in range(lookback, len(arr)):
        X.append(arr[i - lookback:i]); y.append(arr[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

train_arr = train_scaled.values.astype(np.float32)
X_seq, y_seq = make_seq2one(train_arr, LOOKBACK)
X_seq_t, y_seq_t = torch.from_numpy(X_seq), torch.from_numpy(y_seq)
n_val = max(1, int(0.1 * len(X_seq_t)))
X_seq_tr, y_seq_tr   = X_seq_t[:-n_val], y_seq_t[:-n_val]
X_seq_val, y_seq_val = X_seq_t[-n_val:], y_seq_t[-n_val:]
print(f"LSTM training samples: {X_seq_tr.shape[0]}, window={LOOKBACK}, features={n_features}")

lstm_model = LSTMForecaster(n_features)
lstm_model = train_model(lstm_model, X_seq_tr, y_seq_tr, X_seq_val, y_seq_val,
                          epochs=120, patience=15, name="LSTM")


In [ ]:
# Recursive 120-hour forecast
window = train_arr[-LOOKBACK:].copy()
lstm_preds_scaled = []
lstm_model.eval()
with torch.no_grad():
    for h in range(HORIZON):
        x_in = torch.from_numpy(window).unsqueeze(0).to(device)
        nxt = lstm_model(x_in)[0].cpu().numpy()
        nxt[calendar_idx] = future_calendar_scaled[h]   # inject known calendar
        lstm_preds_scaled.append(nxt)
        window = np.vstack([window[1:], nxt])

lstm_preds   = np.array(lstm_preds_scaled) * std.values + mean.values
lstm_pred_df = pd.DataFrame(lstm_preds, columns=feature_cols, index=test_df.index)
print("PyTorch LSTM 5-day forecast complete.")


## 5. Model 2 — XGBoost (direct multi-horizon, unchanged)

Kept identical to the original notebook as the strong classical baseline: one model per
parameter, trained on lag features + lead time `h` + known future calendar at the target
hour. Direct forecasting avoids recursive drift over 120 steps.

In [ ]:
ORIGIN_LAGS = [1, 2, 3, 6, 12, 24, 48, 72]

def make_direct_training(scaled, target_cols, calendar_cols, lags, horizon, origin_step=4):
    n, max_lag = len(scaled), max(lags)
    rows = {c: [] for c in target_cols}
    feats = []
    for origin in range(max_lag, n - horizon, origin_step):
        base = {f"{c}_lag{L}": scaled[c].iloc[origin - L] for c in target_cols for L in lags}
        for h in range(1, horizon + 1):
            row = dict(base); row["lead_h"] = h
            for cc in calendar_cols:
                row[cc] = scaled[cc].iloc[origin + h]
            feats.append(row)
            for c in target_cols:
                rows[c].append(scaled[c].iloc[origin + h])
    return pd.DataFrame(feats), {c: np.array(rows[c]) for c in target_cols}

X_direct, Y_direct = make_direct_training(train_scaled, target_cols, calendar_cols,
                                           ORIGIN_LAGS, HORIZON, origin_step=4)
feat_order = list(X_direct.columns)
print(f"Direct training rows: {X_direct.shape[0]:,}  features: {X_direct.shape[1]}")

xgb_models = {}
for c in target_cols:
    m = xgb.XGBRegressor(
        n_estimators=180, max_depth=5, learning_rate=0.06,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED,
        n_jobs=4, objective="reg:squarederror", tree_method="hist",
    )
    m.fit(X_direct, Y_direct[c])
    xgb_models[c] = m
print(f"Trained {len(xgb_models)} direct XGBoost models.")


In [ ]:
origin_idx = len(train_scaled) - 1
base_row = {f"{c}_lag{L}": train_scaled[c].iloc[origin_idx - (L - 1)]
            for c in target_cols for L in ORIGIN_LAGS}

pred_rows = []
for h in range(1, HORIZON + 1):
    ts = test_df.index[h - 1]
    row = dict(base_row); row["lead_h"] = h
    for cc in calendar_cols:
        row[cc] = full_scaled.loc[ts, cc]
    pred_rows.append(row)
X_fore = pd.DataFrame(pred_rows)[feat_order]

xgb_scaled_full = pd.DataFrame(index=test_df.index, columns=feature_cols, dtype=float)
for c in target_cols:
    xgb_scaled_full[c] = xgb_models[c].predict(X_fore)
for cc in calendar_cols:
    xgb_scaled_full[cc] = full_scaled[cc].iloc[-HORIZON:].values

xgb_preds   = xgb_scaled_full.values * std.values + mean.values
xgb_pred_df = pd.DataFrame(xgb_preds, columns=feature_cols, index=test_df.index)
print("XGBoost 5-day forecast complete.")


## 6. Model 3 — N-BEATS (pure PyTorch)

A **multivariate, direct multi-horizon** adaptation of N-BEATS: instead of one univariate
model per series, a single global model takes the flattened `(lookback, n_features)`
window and outputs the full `(horizon, n_targets)` forecast in one shot — no recursion,
so no compounding drift. Architecture: a doubly-residual stack of fully-connected
"generic" blocks (Oreshkin et al., 2020). Each block consumes the current residual,
predicts a **backcast** (subtracted from the residual passed to the next block) and a
**forecast** (added to a running total) — this lets later blocks focus on what earlier
blocks couldn't explain.

In [ ]:
class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_back, theta_fore, hidden=128, n_layers=4):
        super().__init__()
        layers, d = [], input_size
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.ReLU()]
            d = hidden
        self.fc = nn.Sequential(*layers)
        self.back_head = nn.Linear(hidden, theta_back)
        self.fore_head = nn.Linear(hidden, theta_fore)

    def forward(self, x):
        h = self.fc(x)
        return self.back_head(h), self.fore_head(h)


class NBeats(nn.Module):
    # Generic N-BEATS, adapted to multivariate direct multi-horizon output.
    def __init__(self, lookback, n_features, horizon, n_targets, n_stacks=3, n_blocks=2, hidden=128):
        super().__init__()
        self.horizon, self.n_targets = horizon, n_targets
        input_size = lookback * n_features
        fore_size  = horizon * n_targets
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, input_size, fore_size, hidden=hidden)
            for _ in range(n_stacks * n_blocks)
        ])

    def forward(self, x):
        b = x.shape[0]
        residual = x.reshape(b, -1)
        forecast = torch.zeros(b, self.horizon * self.n_targets, device=x.device)
        for block in self.blocks:
            back, fore = block(residual)
            residual = residual - back
            forecast = forecast + fore
        return forecast.reshape(b, self.horizon, self.n_targets)


### Direct multi-horizon training windows

N-BEATS and N-HiTS both consume `(lookback, n_features)` windows and predict the full
`(horizon, n_targets)` block directly, so we build a dense set of sliding-window
(origin, target) pairs from the training data (every origin, not subsampled — these
models train fast enough on CPU that this isn't a bottleneck).

In [ ]:
def make_direct_windows(scaled_df, lookback, horizon, target_cols_idx):
    arr = scaled_df.values.astype(np.float32)
    X, Y = [], []
    for origin in range(lookback, len(arr) - horizon):
        X.append(arr[origin - lookback:origin])
        Y.append(arr[origin:origin + horizon][:, target_cols_idx])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_np, Y_np = make_direct_windows(train_scaled, LOOKBACK, HORIZON, target_idx)
X_t, Y_t = torch.from_numpy(X_np), torch.from_numpy(Y_np)
n_val_d = max(1, int(0.1 * len(X_t)))
X_tr_d, Y_tr_d   = X_t[:-n_val_d], Y_t[:-n_val_d]
X_val_d, Y_val_d = X_t[-n_val_d:], Y_t[-n_val_d:]
print(f"Direct-horizon training windows: {X_tr_d.shape[0]} train / {X_val_d.shape[0]} val")

last_window = torch.from_numpy(train_scaled.values[-LOOKBACK:].astype(np.float32)).unsqueeze(0)


In [ ]:
nbeats = NBeats(LOOKBACK, n_features, HORIZON, n_targets, n_stacks=3, n_blocks=2, hidden=128)
nbeats = train_model(nbeats, X_tr_d, Y_tr_d, X_val_d, Y_val_d, epochs=200, patience=20, name="N-BEATS")

with torch.no_grad():
    nbeats_pred_scaled = nbeats(last_window.to(device))[0].cpu().numpy()
nbeats_preds_real = nbeats_pred_scaled * std[target_cols].values + mean[target_cols].values
nbeats_pred_df = pd.DataFrame(nbeats_preds_real, columns=target_cols, index=test_df.index)
print("N-BEATS 5-day forecast complete.")


## 7. Model 4 — N-HiTS (pure PyTorch)

N-HiTS (Challu et al., 2023) was purpose-built for **long-horizon** forecasting — exactly
this problem (120-step / 5-day-ahead). It generalizes N-BEATS with two ideas:

1. **Multi-rate input sampling**: each stack first pools (downsamples) the input window
   at a different rate, so different stacks naturally specialize in different timescales.
   Here the rates are **24h, 12h, 1h** — matched to this dataset's diurnal cycle, the
   ~12h semidiurnal tide, and fine-grained residual structure.
2. **Hierarchical interpolation**: each stack predicts a coarse forecast basis at a small
   number of points, then linearly interpolates it up to the full horizon — this is far
   more parameter-efficient than predicting all 120 steps directly per stack, which is
   what makes N-HiTS scale well to long horizons.

In [ ]:
class NHitsBlock(nn.Module):
    def __init__(self, lookback, n_features, horizon, n_targets, pool_kernel, hidden=128, theta_ratio=4):
        super().__init__()
        self.lookback, self.n_features = lookback, n_features
        self.horizon, self.n_targets = horizon, n_targets
        self.pool_kernel = pool_kernel
        pooled_len = lookback // pool_kernel
        theta_fore_len = max(2, horizon // theta_ratio)
        self.theta_fore_len = theta_fore_len
        self.fc = nn.Sequential(
            nn.Linear(pooled_len * n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.back_head = nn.Linear(hidden, lookback * n_features)
        self.fore_head = nn.Linear(hidden, theta_fore_len * n_targets)

    def forward(self, x):
        b = x.shape[0]
        pooled = F.avg_pool1d(x.transpose(1, 2), self.pool_kernel).transpose(1, 2).reshape(b, -1)
        h = self.fc(pooled)
        back = self.back_head(h).reshape(b, self.lookback, self.n_features)
        theta_fore = self.fore_head(h).reshape(b, self.theta_fore_len, self.n_targets)
        # hierarchical interpolation: coarse basis -> full horizon resolution
        fore = F.interpolate(theta_fore.transpose(1, 2), size=self.horizon,
                              mode="linear", align_corners=False).transpose(1, 2)
        return back, fore


class NHits(nn.Module):
    def __init__(self, lookback, n_features, horizon, n_targets,
                 pool_kernels=(24, 12, 1), hidden=128, theta_ratio=4):
        super().__init__()
        self.horizon, self.n_targets = horizon, n_targets
        self.blocks = nn.ModuleList([
            NHitsBlock(lookback, n_features, horizon, n_targets, k, hidden=hidden, theta_ratio=theta_ratio)
            for k in pool_kernels
        ])

    def forward(self, x):
        residual, b = x, x.shape[0]
        forecast = torch.zeros(b, self.horizon, self.n_targets, device=x.device)
        for block in self.blocks:
            back, fore = block(residual)
            residual = residual - back
            forecast = forecast + fore
        return forecast


In [ ]:
nhits = NHits(LOOKBACK, n_features, HORIZON, n_targets, pool_kernels=(24, 12, 1),
              hidden=128, theta_ratio=4)
nhits = train_model(nhits, X_tr_d, Y_tr_d, X_val_d, Y_val_d, epochs=200, patience=20, name="N-HiTS")

with torch.no_grad():
    nhits_pred_scaled = nhits(last_window.to(device))[0].cpu().numpy()
nhits_preds_real = nhits_pred_scaled * std[target_cols].values + mean[target_cols].values
nhits_pred_df = pd.DataFrame(nhits_preds_real, columns=target_cols, index=test_df.index)
print("N-HiTS 5-day forecast complete.")


## 8. Model 5 — DLinear (Zeng et al., AAAI 2023)

From *"Are Transformers Effective for Time Series Forecasting?"* — decomposes each
target series into a slow **trend** (moving average) and a **seasonal** residual, then
applies one linear layer per component mapping the 72-hour lookback directly to the
120-hour horizon (channel-independent: each parameter forecast from its own history
only, with shared linear weights across parameters). Deliberately the simplest model in
this notebook — a critical sanity-check baseline against which the deep architectures
must earn their extra complexity.

In [ ]:
class MovingAvg(nn.Module):
    """Centered moving average with edge-replicate padding (keeps length)."""
    def __init__(self, kernel_size):
        super().__init__()
        self.k = kernel_size
        self.avg = nn.AvgPool1d(kernel_size, stride=1, padding=0)

    def forward(self, x):  # x: (b, c, l)
        front = x[:, :, 0:1].repeat(1, 1, (self.k - 1) // 2)
        end = x[:, :, -1:].repeat(1, 1, self.k - (self.k - 1) // 2 - 1)
        return self.avg(torch.cat([front, x, end], dim=2))


class DLinear(nn.Module):
    def __init__(self, lookback, horizon, target_idx, kernel_size=13):
        super().__init__()
        self.decomp = MovingAvg(kernel_size)
        self.lin_trend = nn.Linear(lookback, horizon)
        self.lin_seasonal = nn.Linear(lookback, horizon)
        self.target_idx = target_idx

    def forward(self, x):  # x: (b, lookback, n_features)
        xt = x[:, :, self.target_idx].transpose(1, 2)        # (b, n_targets, lookback)
        trend = self.decomp(xt)
        seasonal = xt - trend
        out = self.lin_trend(trend) + self.lin_seasonal(seasonal)
        return out.transpose(1, 2)                            # (b, horizon, n_targets)


In [ ]:
dlinear = DLinear(LOOKBACK, HORIZON, target_idx)
dlinear = train_model(dlinear, X_tr_d, Y_tr_d, X_val_d, Y_val_d, epochs=150, patience=20, name="DLinear")

with torch.no_grad():
    dlinear_pred_scaled = dlinear(last_window.to(device))[0].cpu().numpy()
dlinear_preds_real = dlinear_pred_scaled * std[target_cols].values + mean[target_cols].values
dlinear_pred_df = pd.DataFrame(dlinear_preds_real, columns=target_cols, index=test_df.index)
print("DLinear 5-day forecast complete.")


## 9. Model 6 — TiDE (Das et al., Google, TMLR 2023)

*Time-series Dense Encoder*: a dense (MLP-only) encoder compresses the lookback window
of target values **together with the known future calendar covariates** into a small
latent vector; a dense decoder expands that latent directly to the `(horizon, n_targets)`
forecast. A **global linear residual** (a per-channel lookback→horizon linear "highway",
in the spirit of DLinear) is added on top so the dense layers only need to model what the
linear part misses. TiDE's headline result is matching Transformer-level long-horizon
accuracy at a fraction of the training cost — and it's one of the few mainstream
architectures designed to use known-future covariates natively, which fits this problem
well since hour-of-day / day-of-year are known exactly in advance.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
        self.proj = nn.Linear(d_in, d_out) if d_in != d_out else nn.Identity()
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_out)

    def forward(self, x):
        h = self.fc2(self.drop(F.relu(self.fc1(x))))
        return self.norm(h + self.proj(x))


class TiDE(nn.Module):
    def __init__(self, lookback, n_targets, n_calendar, horizon, target_idx, hidden=128, latent=64):
        super().__init__()
        self.horizon, self.n_targets, self.target_idx = horizon, n_targets, target_idx
        enc_in = lookback * n_targets + horizon * n_calendar
        self.encoder = nn.Sequential(ResBlock(enc_in, hidden, hidden), ResBlock(hidden, hidden, latent))
        self.decoder = nn.Sequential(ResBlock(latent, hidden, hidden), nn.Linear(hidden, horizon * n_targets))
        self.global_residual = nn.Linear(lookback, horizon)   # DLinear-style highway connection

    def forward(self, x, future_calendar):
        # x: (b, lookback, n_features) | future_calendar: (b, horizon, n_calendar), known in advance
        b = x.shape[0]
        tgt_hist = x[:, :, self.target_idx]
        enc_in = torch.cat([tgt_hist.reshape(b, -1), future_calendar.reshape(b, -1)], dim=1)
        latent = self.encoder(enc_in)
        dec_out = self.decoder(latent).reshape(b, self.horizon, self.n_targets)
        residual = self.global_residual(tgt_hist.transpose(1, 2)).transpose(1, 2)
        return dec_out + residual


In [ ]:
def make_future_calendar(scaled_df, lookback, horizon, calendar_cols_idx):
    arr = scaled_df.values.astype(np.float32)
    C = [arr[o:o + horizon][:, calendar_cols_idx] for o in range(lookback, len(arr) - horizon)]
    return np.array(C, dtype=np.float32)

C_np = make_future_calendar(train_scaled, LOOKBACK, HORIZON, calendar_idx)
C_t = torch.from_numpy(C_np)
C_tr_d, C_val_d = C_t[:-n_val_d], C_t[-n_val_d:]
future_cal_last = torch.from_numpy(
    full_scaled[calendar_cols].iloc[-HORIZON:].values.astype(np.float32)
).unsqueeze(0)

def train_tide(model, X_tr, C_tr, Y_tr, X_val, C_val, Y_val, epochs=150, batch_size=64,
                lr=1e-3, patience=20):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=6)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait = float("inf"), None, 0
    n = len(X_tr); t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            xb, cb, yb = X_tr[b].to(device), C_tr[b].to(device), Y_tr[b].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb, cb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val.to(device), C_val.to(device)), Y_val.to(device)).item()
        sched.step(val_loss)
        if val_loss < best_val - 1e-5:
            best_val, wait = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"{'TiDE':10s} best_val_loss={best_val:.4f}  epochs_run={ep+1:3d}  time={time.time()-t0:5.1f}s")
    return model

tide = TiDE(LOOKBACK, n_targets, len(calendar_cols), HORIZON, target_idx)
tide = train_tide(tide, X_tr_d, C_tr_d, Y_tr_d, X_val_d, C_val_d, Y_val_d)

with torch.no_grad():
    tide_pred_scaled = tide(last_window.to(device), future_cal_last.to(device))[0].cpu().numpy()
tide_preds_real = tide_pred_scaled * std[target_cols].values + mean[target_cols].values
tide_pred_df = pd.DataFrame(tide_preds_real, columns=target_cols, index=test_df.index)
print("TiDE 5-day forecast complete.")


## 10. Model 7 — TSMixer (Chen et al., Google, 2023)

*"TSMixer: An All-MLP Architecture for Time Series Forecasting"* alternates two kinds of
MLP blocks, in the spirit of the MLP-Mixer vision architecture: **time-mixing** (an MLP
applied along the time axis, shared across all 21 input channels — learns temporal
patterns) and **feature-mixing** (an MLP applied across channels at each timestep —
learns cross-parameter relationships such as wind → wave growth or pressure → wind). Both
use residual connections + LayerNorm. This is the one model here that explicitly learns
**cross-parameter coupling**, which matters in this dataset (wind drives waves with a lag,
pressure gradients drive wind, temperature and dew point are linked, etc.).

In [ ]:
class TSMixerBlock(nn.Module):
    def __init__(self, lookback, n_channels, hidden_time=64, hidden_feat=64, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_channels)
        self.time_mix = nn.Sequential(nn.Linear(lookback, hidden_time), nn.ReLU(),
                                       nn.Dropout(dropout), nn.Linear(hidden_time, lookback))
        self.norm2 = nn.LayerNorm(n_channels)
        self.feat_mix = nn.Sequential(nn.Linear(n_channels, hidden_feat), nn.ReLU(),
                                       nn.Dropout(dropout), nn.Linear(hidden_feat, n_channels))

    def forward(self, x):  # x: (b, lookback, n_channels)
        y = self.norm1(x).transpose(1, 2)
        x = x + self.time_mix(y).transpose(1, 2)             # time-mixing, residual
        x = x + self.feat_mix(self.norm2(x))                  # feature-mixing, residual
        return x


class TSMixer(nn.Module):
    def __init__(self, lookback, n_features, horizon, target_idx, n_blocks=3, hidden_time=64, hidden_feat=64):
        super().__init__()
        self.target_idx = target_idx
        self.blocks = nn.ModuleList([
            TSMixerBlock(lookback, n_features, hidden_time, hidden_feat) for _ in range(n_blocks)
        ])
        self.head = nn.Linear(lookback, horizon)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        out = self.head(x.transpose(1, 2)).transpose(1, 2)    # (b, horizon, n_features)
        return out[:, :, self.target_idx]


In [ ]:
tsmixer = TSMixer(LOOKBACK, n_features, HORIZON, target_idx)
tsmixer = train_model(tsmixer, X_tr_d, Y_tr_d, X_val_d, Y_val_d, epochs=150, patience=20, name="TSMixer")

with torch.no_grad():
    tsmixer_pred_scaled = tsmixer(last_window.to(device))[0].cpu().numpy()
tsmixer_preds_real = tsmixer_pred_scaled * std[target_cols].values + mean[target_cols].values
tsmixer_pred_df = pd.DataFrame(tsmixer_preds_real, columns=target_cols, index=test_df.index)
print("TSMixer 5-day forecast complete.")


## 11. Model 8 — Harmonic Regression + Residual MLP Hybrid (domain-specific)

`MARINE_FORECASTING_IMPLEMENTATION_GUIDE.md` (§1.1, Tidal Level) recommends exactly this
recipe from the recent literature (Zhang et al. 2024, VMD-LSTM for tidal level): decompose
each series into a **deterministic periodic part** + a **residual learned by ML**.

1. **Harmonic regression** (closed-form least squares): fit each target parameter against
   sin/cos terms at the physically meaningful periods in this domain — **24h** (diurnal),
   **12h** (semidiurnal/synoptic), **12.42h** (M2 tide), **23.93h** (K1), **25.82h** (O1) —
   plus a slow linear trend. Because these periods are known and fixed, the harmonic
   component can be evaluated at *any* future timestamp with **zero error growth** —
   it doesn't degrade with forecast horizon the way learned models do.
2. **Residual MLP**: train a small dense direct-horizon network only on what's left over
   (storm surges, weather noise, anything non-periodic) — a much easier learning problem
   than the raw series.
3. **Recompose**: final forecast = harmonic future + predicted residual.

This should win decisively on **tidal level** (it's the textbook use case) and help on
other periodic series (current speed, solar radiation, diurnal temperature), while adding
little on parameters with no strong periodicity (precipitation, storm-driven wind).

In [ ]:
PERIODS_HOURS = [24.0, 12.0, 12.42, 23.93, 25.82, 6.0]   # diurnal, synoptic, M2, K1, O1, 6-day synoptic

t_all = np.arange(len(model_data))
design_cols = [np.ones_like(t_all, dtype=float)]
for P in PERIODS_HOURS:
    design_cols.append(np.sin(2 * np.pi * t_all / P))
    design_cols.append(np.cos(2 * np.pi * t_all / P))
design_cols.append(t_all / len(t_all))                       # slow linear trend term
Phi = np.stack(design_cols, axis=1)
Phi_train = Phi[:len(train_df)]

harmonic_full = pd.DataFrame(index=model_data.index, columns=target_cols, dtype=float)
residual_train = pd.DataFrame(index=train_df.index, columns=target_cols, dtype=float)
for c in target_cols:
    y_train = train_df[c].values
    coef, *_ = np.linalg.lstsq(Phi_train, y_train, rcond=None)
    harmonic_full[c] = Phi @ coef
    residual_train[c] = y_train - Phi_train @ coef

print("Harmonic regression fit for", len(target_cols), "parameters.")
print("Example R^2 (tidal_level_m):",
      round(1 - residual_train["tidal_level_m"].var() / train_df["tidal_level_m"].var(), 3))


In [ ]:
resid_mean = residual_train.mean()
resid_std  = residual_train.std().replace(0, 1)
residual_train_scaled = (residual_train - resid_mean) / resid_std

resid_model_data = residual_train_scaled.copy()
for cc in calendar_cols:
    resid_model_data[cc] = train_scaled[cc].values
resid_feature_cols = target_cols + calendar_cols
resid_target_idx = [resid_feature_cols.index(c) for c in target_cols]

Xr_np, Yr_np = make_direct_windows(resid_model_data, LOOKBACK, HORIZON, resid_target_idx)
Xr_t, Yr_t = torch.from_numpy(Xr_np), torch.from_numpy(Yr_np)
n_val_r = max(1, int(0.1 * len(Xr_t)))
Xr_tr, Yr_tr   = Xr_t[:-n_val_r], Yr_t[:-n_val_r]
Xr_val, Yr_val = Xr_t[-n_val_r:], Yr_t[-n_val_r:]

class ResidualMLP(nn.Module):
    """Small dense direct-horizon model for the post-harmonic residual series."""
    def __init__(self, lookback, n_in, horizon, n_targets, hidden=128):
        super().__init__()
        self.horizon, self.n_targets = horizon, n_targets
        self.net = nn.Sequential(
            nn.Linear(lookback * n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, horizon * n_targets),
        )

    def forward(self, x):
        b = x.shape[0]
        return self.net(x.reshape(b, -1)).reshape(b, self.horizon, self.n_targets)

resid_model = ResidualMLP(LOOKBACK, len(resid_feature_cols), HORIZON, n_targets)
resid_model = train_model(resid_model, Xr_tr, Yr_tr, Xr_val, Yr_val, epochs=150, patience=20, name="HarmResid")

last_resid_window = torch.from_numpy(resid_model_data.values[-LOOKBACK:].astype(np.float32)).unsqueeze(0)
with torch.no_grad():
    resid_pred_scaled = resid_model(last_resid_window.to(device))[0].cpu().numpy()
resid_pred_real   = resid_pred_scaled * resid_std[target_cols].values + resid_mean[target_cols].values
harmonic_future   = harmonic_full[target_cols].iloc[-HORIZON:].values
hybrid_preds_real = harmonic_future + resid_pred_real
hybrid_pred_df = pd.DataFrame(hybrid_preds_real, columns=target_cols, index=test_df.index)
print("Harmonic-Residual Hybrid 5-day forecast complete.")


## 12. Model 9 — iTransformer (Liu et al., 2023)

Standard Transformers for time series tokenize each **timestep** and attend across time.
iTransformer inverts this: each **variate** (parameter) becomes a token — its entire
72-hour lookback window is embedded into a single vector — and self-attention runs
**across the 21 variates**, directly learning which parameters inform which (wind→wave,
pressure→wind, temperature↔dew point), in a more flexible, learned way than TSMixer's
fixed feature-mixing MLP. A feed-forward head then maps each variate's token straight to
its 120-hour forecast — no decoder, no autoregression.

The literature reports this design is both more accurate *and* faster to train than a
standard Transformer on high-variate data, because attention runs over a (small) number
of variates rather than a (long) number of timesteps.

In [ ]:
class ITransformer(nn.Module):
    def __init__(self, lookback, n_features, horizon, target_idx, d_model=64, n_heads=4,
                 n_layers=2, dropout=0.1):
        super().__init__()
        self.target_idx = target_idx
        self.embed = nn.Linear(lookback, d_model)        # "inverted" embedding: whole series -> 1 token
        self.var_id = nn.Parameter(torch.randn(n_features, d_model) * 0.02)   # learnable variate identity
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 2,
                                            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):                                  # x: (b, lookback, n_features)
        tok = self.embed(x.transpose(1, 2))                # (b, n_features, d_model) -- each variate = 1 token
        tok = tok + self.var_id.unsqueeze(0)
        tok = self.encoder(tok)                             # self-attention ACROSS variates
        out = self.head(tok)                                # (b, n_features, horizon)
        return out.transpose(1, 2)[:, :, self.target_idx]   # (b, horizon, n_targets)


In [ ]:
itransformer = ITransformer(LOOKBACK, n_features, HORIZON, target_idx, d_model=64, n_heads=4, n_layers=2)
itransformer = train_model(itransformer, X_tr_d, Y_tr_d, X_val_d, Y_val_d, epochs=200, patience=25, name="iTransformer")

with torch.no_grad():
    itransformer_pred_scaled = itransformer(last_window.to(device))[0].cpu().numpy()
itransformer_preds_real = itransformer_pred_scaled * std[target_cols].values + mean[target_cols].values
itransformer_pred_df = pd.DataFrame(itransformer_preds_real, columns=target_cols, index=test_df.index)
print("iTransformer 5-day forecast complete.")


## 13. Model 10 — PatchTST (Nie et al., ICLR 2023)

PatchTST applies two ideas that the recent literature repeatedly credits for
Transformer gains on time series:

1. **Patching**: instead of attending over 72 individual hourly points (mostly
   redundant, noisy), the lookback window is split into 6 patches of 12 hours each —
   each patch is embedded into one token, giving the attention mechanism a much shorter,
   more informative sequence to work with (and roughly 12× fewer tokens than per-timestep
   attention).
2. **Channel independence**: the *same* Transformer weights are applied to every one of
   the 21 channels independently (each channel's 6 patches form its own token sequence).
   This effectively multiplies the training data 21×, which matters a lot on a dataset
   this size, and prevents one noisy channel's patterns from corrupting another's
   weights — at the cost of not modeling cross-channel relationships at all (that's
   exactly what iTransformer and TSMixer are for).

In [ ]:
class PatchTST(nn.Module):
    def __init__(self, lookback, n_features, horizon, target_idx, patch_len=12, d_model=64,
                 n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        assert lookback % patch_len == 0
        self.n_patches = lookback // patch_len
        self.target_idx = target_idx
        self.patch_len = patch_len
        self.patch_embed = nn.Linear(patch_len, d_model)     # SHARED across all channels (channel-independent)
        self.pos = nn.Parameter(torch.randn(self.n_patches, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 2,
                                            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(self.n_patches * d_model, horizon)

    def forward(self, x):                          # x: (b, lookback, n_features)
        b, L, C = x.shape
        xt = x.transpose(1, 2).reshape(b * C, self.n_patches, self.patch_len)
        tok = self.patch_embed(xt) + self.pos.unsqueeze(0)   # (b*C, n_patches, d_model)
        tok = self.encoder(tok)                               # attention ACROSS time-patches, per channel
        out = self.head(tok.reshape(b * C, -1)).reshape(b, C, -1)   # (b, n_features, horizon)
        return out.transpose(1, 2)[:, :, self.target_idx]


In [ ]:
patchtst = PatchTST(LOOKBACK, n_features, HORIZON, target_idx, patch_len=12, d_model=64, n_heads=4, n_layers=2)
patchtst = train_model(patchtst, X_tr_d, Y_tr_d, X_val_d, Y_val_d, epochs=150, patience=20, name="PatchTST")

with torch.no_grad():
    patchtst_pred_scaled = patchtst(last_window.to(device))[0].cpu().numpy()
patchtst_preds_real = patchtst_pred_scaled * std[target_cols].values + mean[target_cols].values
patchtst_pred_df = pd.DataFrame(patchtst_preds_real, columns=target_cols, index=test_df.index)
print("PatchTST 5-day forecast complete.")


## 14. Model 11 — DeepAR (Salinas et al., Amazon)

Every model so far predicts one number per hour. DeepAR predicts a **distribution**: at
each step the RNN outputs `(μ, σ)` for every parameter, trained by minimizing Gaussian
negative log-likelihood instead of MSE. To forecast 120 hours we don't roll forward a
single point estimate — we draw **100 independent sample paths**, each one sampling from
the predicted Gaussian at every step and feeding that *sampled* value back in (true
ancestral sampling, not just the mean). The point forecast is the mean across the 100
paths; the spread across paths is a genuine, **horizon-growing** uncertainty band — the
one capability this notebook was missing entirely until now.

In [ ]:
class DeepAR(nn.Module):
    def __init__(self, n_features, hidden1=96, hidden2=48, dropout=0.2):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, hidden1, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.drop2 = nn.Dropout(dropout)
        self.mu_head = nn.Linear(hidden2, n_features)
        self.logsigma_head = nn.Linear(hidden2, n_features)

    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        _, (h, _) = self.lstm2(x)
        h = self.drop2(h[-1])
        mu = self.mu_head(h)
        log_sigma = self.logsigma_head(h).clamp(-6, 3)
        return mu, log_sigma


def gaussian_nll(mu, log_sigma, target):
    sigma = torch.exp(log_sigma)
    return (0.5 * ((target - mu) / sigma) ** 2 + log_sigma).mean()


def train_deepar(model, X_tr, y_tr, X_val, y_val, epochs=120, batch_size=32, lr=1e-3, patience=15):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=6)
    best_val, best_state, wait = float("inf"), None, 0
    n = len(X_tr); t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            xb, yb = X_tr[b].to(device), y_tr[b].to(device)
            opt.zero_grad()
            mu, log_sigma = model(xb)
            loss = gaussian_nll(mu, log_sigma, yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            mu, log_sigma = model(X_val.to(device))
            val_loss = gaussian_nll(mu, log_sigma, y_val.to(device)).item()
        sched.step(val_loss)
        if val_loss < best_val - 1e-4:
            best_val, wait = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    print(f"{'DeepAR':10s} best_val_nll={best_val:.4f}  epochs_run={ep+1:3d}  time={time.time()-t0:5.1f}s")
    return model

deepar = DeepAR(n_features)
deepar = train_deepar(deepar, X_seq_tr, y_seq_tr, X_seq_val, y_seq_val)


In [ ]:
N_MC_SAMPLES = 100
deepar.eval()
all_paths = np.zeros((N_MC_SAMPLES, HORIZON, n_features), dtype=np.float32)
with torch.no_grad():
    for s in range(N_MC_SAMPLES):
        window = train_arr[-LOOKBACK:].copy()
        for h in range(HORIZON):
            x_in = torch.from_numpy(window).unsqueeze(0).to(device)
            mu, log_sigma = deepar(x_in)
            mu_np, sigma_np = mu[0].cpu().numpy(), torch.exp(log_sigma)[0].cpu().numpy()
            sample = mu_np + sigma_np * np.random.randn(n_features).astype(np.float32)
            sample[calendar_idx] = future_calendar_scaled[h]
            all_paths[s, h] = sample
            window = np.vstack([window[1:], sample])

deepar_mean_scaled = all_paths.mean(axis=0)
deepar_std_scaled = all_paths.std(axis=0)
deepar_mean_real = deepar_mean_scaled[:, target_idx] * std[target_cols].values + mean[target_cols].values
deepar_std_real = deepar_std_scaled[:, target_idx] * std[target_cols].values
deepar_pred_df = pd.DataFrame(deepar_mean_real, columns=target_cols, index=test_df.index)
deepar_std_df = pd.DataFrame(deepar_std_real, columns=target_cols, index=test_df.index)
print(f"DeepAR: {N_MC_SAMPLES} Monte-Carlo sample paths complete.")
print("Predicted std growth (tidal_level_m): h=1 ->", round(deepar_std_df['tidal_level_m'].iloc[0], 4),
      " h=120 ->", round(deepar_std_df['tidal_level_m'].iloc[-1], 4), "(should grow with horizon)")


## 15. Model 12 — Gaussian Process (GPyTorch)

A Bayesian model fit **independently per parameter** (17 small GPs, not one global
model) with an additive kernel built from domain knowledge:

`k = k_diurnal(24h) + k_tidal(12.42h) + k_local(RBF)`

Both periodic kernels' periods are **frozen at the true known values** rather than
learned — letting gradient descent search for the period as a free parameter turned out
to actively hurt: it would wander off the true 24h/12.42h cycle into a worse local
optimum, collapsing both periodic kernels' output scale toward zero and leaving the GP
to revert to a flat mean almost immediately past the training window. Freezing the
period and only learning each kernel's outputscale/lengthscale fixed this completely.
Periodic kernels (unlike RBF) extrapolate their cycle **indefinitely**, which is exactly
the property forecasting needs — RBF alone decays to the prior mean past its
lengthscale. Fit on the last 30 days of training data per parameter (bounds the O(n³)
exact-GP cost); produces a calibrated posterior mean **and** 95% credible interval with
no sampling required.

In [ ]:
import gpytorch

class MarineGP(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.diurnal_kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.PeriodicKernel())
        self.tidal_kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.PeriodicKernel())
        self.local_kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
        self.covar_module = self.diurnal_kernel + self.tidal_kernel + self.local_kernel

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean_module(x), self.covar_module(x))


GP_WINDOW_HOURS = 24 * 30   # last 30 days of train -- bounds exact-GP O(n^3) cost
gp_pred = {}
gp_std = {}
t0 = time.time()
for c in target_cols:
    y_train_full = train_df[c].iloc[-GP_WINDOW_HOURS:].values
    t_train = np.arange(len(y_train_full), dtype=np.float64)
    y_mean_c, y_std_c = y_train_full.mean(), y_train_full.std() + 1e-8
    t_scale = t_train.std()

    train_x = torch.tensor(t_train / t_scale, dtype=torch.float32).unsqueeze(-1)
    train_y = torch.tensor((y_train_full - y_mean_c) / y_std_c, dtype=torch.float32)

    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    gp_model = MarineGP(train_x, train_y, likelihood)
    gp_model.diurnal_kernel.base_kernel.initialize(period_length=24.0 / t_scale)
    gp_model.tidal_kernel.base_kernel.initialize(period_length=12.42 / t_scale)
    gp_model.diurnal_kernel.base_kernel.raw_period_length.requires_grad = False
    gp_model.tidal_kernel.base_kernel.raw_period_length.requires_grad = False

    gp_model.train(); likelihood.train()
    opt = torch.optim.Adam(gp_model.parameters(), lr=0.05)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, gp_model)
    for _ in range(150):
        opt.zero_grad()
        loss = -mll(gp_model(train_x), train_y)
        loss.backward(); opt.step()

    gp_model.eval(); likelihood.eval()
    t_future = np.arange(len(y_train_full), len(y_train_full) + HORIZON, dtype=np.float64)
    test_x = torch.tensor(t_future / t_scale, dtype=torch.float32).unsqueeze(-1)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(gp_model(test_x))
        gp_pred[c] = pred.mean.numpy() * y_std_c + y_mean_c
        gp_std[c] = pred.stddev.numpy() * y_std_c

gp_pred_df = pd.DataFrame(gp_pred, index=test_df.index)[target_cols]
gp_std_df = pd.DataFrame(gp_std, index=test_df.index)[target_cols]
print(f"Gaussian Process: {len(target_cols)} per-parameter models fit. time={time.time()-t0:.1f}s")


## 16. Reconstruct physical values & score all twelve models

Convert `sin`/`cos` wind-direction predictions back to degrees, then compute **MAE**,
**RMSE**, and **skill vs persistence** (holding the last observation constant) for every
model and parameter.

In [ ]:
def reconstruct(pred_df):
    out = pred_df.copy()
    out["wind_direction_deg"] = (np.rad2deg(np.arctan2(out["wind_dir_sin"], out["wind_dir_cos"])) % 360)
    return out

lstm_final   = reconstruct(lstm_pred_df)
xgb_final    = reconstruct(xgb_pred_df)
nbeats_final  = reconstruct(nbeats_pred_df)
nhits_final   = reconstruct(nhits_pred_df)
dlinear_final = reconstruct(dlinear_pred_df)
tide_final    = reconstruct(tide_pred_df)
tsmixer_final = reconstruct(tsmixer_pred_df)
hybrid_final  = reconstruct(hybrid_pred_df)
itransformer_final = reconstruct(itransformer_pred_df)
patchtst_final      = reconstruct(patchtst_pred_df)
deepar_final  = reconstruct(deepar_pred_df)
gp_final      = reconstruct(gp_pred_df)
truth         = df.iloc[-HORIZON:].copy()

report_params = [
    "significant_wave_height_m", "wave_period_s", "wind_speed_ms",
    "wind_direction_deg", "tidal_level_m", "current_speed_ms",
    "sea_surface_temp_c", "salinity_psu", "conductivity_mscm",
    "air_pressure_hpa", "air_temp_c", "relative_humidity_pct",
    "dew_point_c", "precipitation_mmh", "solar_radiation_wm2", "visibility_km",
]
MODELS = {
    "LSTM": lstm_final, "XGBoost": xgb_final, "N-BEATS": nbeats_final, "N-HiTS": nhits_final,
    "DLinear": dlinear_final, "TiDE": tide_final, "TSMixer": tsmixer_final,
    "Harmonic-Residual": hybrid_final,
    "iTransformer": itransformer_final, "PatchTST": patchtst_final,
    "DeepAR": deepar_final, "Gaussian Process": gp_final,
}

def circ_mae(true, pred):
    return np.abs((true - pred + 180) % 360 - 180).mean()

last_obs = df.iloc[-HORIZON - 1]
metrics = []
for p in report_params:
    yt = truth[p].values
    yp_persist = np.repeat(last_obs[p], HORIZON)
    is_circular = (p == "wind_direction_deg")
    mae_p = circ_mae(yt, yp_persist) if is_circular else mean_absolute_error(yt, yp_persist)
    row = {"parameter": p, "Persistence_MAE": round(mae_p, 3)}
    best_mae, best_name = np.inf, None
    for name, pred_df in MODELS.items():
        yhat = pred_df[p].values
        if is_circular:
            mae = circ_mae(yt, yhat); rmse = np.nan
        else:
            mae = mean_absolute_error(yt, yhat)
            rmse = np.sqrt(mean_squared_error(yt, yhat))
        skill = (1 - mae / mae_p) * 100 if mae_p > 0 else np.nan
        row[f"{name}_MAE"] = round(mae, 3)
        row[f"{name}_RMSE"] = round(rmse, 3) if rmse == rmse else np.nan
        row[f"{name}_skill_%"] = round(skill, 1)
        if mae < best_mae:
            best_mae, best_name = mae, name
    row["best_model"] = best_name
    metrics.append(row)

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("metrics_summary_pytorch.csv", index=False)
metrics_df


## 17. Visualize the validation

Black = actual (held-out truth); each model gets its own line style. The green line
marks the forecast start.

In [ ]:
hist_tail = df.iloc[-HORIZON - 72:-HORIZON]
key_plots = ["significant_wave_height_m", "wind_speed_ms", "tidal_level_m",
             "current_speed_ms", "air_pressure_hpa", "sea_surface_temp_c",
             "relative_humidity_pct", "wave_period_s", "visibility_km"]
styles = {
    "LSTM": ("tab:blue", "--"), "XGBoost": ("tab:red", ":"),
    "N-BEATS": ("tab:green", "-."), "N-HiTS": ("tab:purple", (0, (3, 1, 1, 1))),
    "DLinear": ("tab:orange", (0, (5, 1))), "TiDE": ("tab:brown", (0, (1, 1))),
    "TSMixer": ("tab:pink", (0, (4, 2, 1, 2))), "Harmonic-Residual": ("tab:cyan", (0, (6, 2))),
    "iTransformer": ("tab:olive", (0, (2, 2, 4, 2))), "PatchTST": ("tab:gray", (0, (1, 1, 3, 1))),
    "DeepAR": ("gold", (0, (3, 3))), "Gaussian Process": ("darkred", (0, (1, 2, 1, 2))),
}

fig, axes = plt.subplots(3, 3, figsize=(19, 13))
for ax, p in zip(axes.ravel(), key_plots):
    ax.plot(hist_tail.index, hist_tail[p], color="0.6", lw=1, label="history")
    ax.plot(truth.index, truth[p], color="black", lw=2.2, label="actual")
    for name, pred_df in MODELS.items():
        c, ls = styles[name]
        ax.plot(truth.index, pred_df[p], color=c, lw=1.3, ls=ls, label=name)
    ax.axvline(truth.index[0], color="green", lw=1, alpha=0.5)
    ax.set_title(p, fontsize=11); ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
axes.ravel()[0].legend(fontsize=8, loc="upper left", ncol=2)
fig.suptitle("Marine 5-Day Forecast: 8 PyTorch/XGBoost Models vs Actual",
             fontsize=14, y=1.0)
fig.tight_layout()
fig.savefig("forecast_plots_pytorch.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
skill_cols = [f"{m}_skill_%" for m in MODELS]
params_lbl = metrics_df["parameter"].str.replace("_", " ").str.title()
y = np.arange(len(params_lbl)); w = 0.8 / len(MODELS)
fig, ax = plt.subplots(figsize=(12, 9))
for i, name in enumerate(MODELS):
    c, _ = styles[name]
    ax.barh(y + (i - 1.5) * w, metrics_df[f"{name}_skill_%"], w, label=name, color=c, alpha=0.85)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(y); ax.set_yticklabels(params_lbl, fontsize=9)
ax.set_xlabel("Forecast skill vs persistence (%) — higher is better")
ax.set_title("5-Day Forecast Skill by Parameter — PyTorch Models", fontsize=13, weight="bold")
ax.legend(); ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig("metrics_comparison_pytorch.png", dpi=120, bbox_inches="tight")
plt.show()

best_counts = metrics_df["best_model"].value_counts()
print("Parameters won per model:")
print(best_counts.to_string())
for name in MODELS:
    print(f"Mean skill -> {name}: {metrics_df[f'{name}_skill_%'].mean():.1f}%")


## 18. Save outputs for the results dashboard

Saves a long-format CSV (actual + all 4 model predictions per parameter per hour) and the
metrics table — these are read directly by the Streamlit dashboard (`app.py`).

In [ ]:
fva = pd.DataFrame({"timestamp": truth.index})
for p in report_params:
    fva[f"{p}__actual"] = truth[p].values
    for name, pred_df in MODELS.items():
        fva[f"{p}__{name.lower().replace('-', '')}"] = pred_df[p].values
fva.to_csv("forecast_vs_actual_pytorch.csv", index=False)

print("Saved: forecast_vs_actual_pytorch.csv, metrics_summary_pytorch.csv,")
print("       forecast_plots_pytorch.png, metrics_comparison_pytorch.png")


In [ ]:
unc = pd.DataFrame({"timestamp": truth.index})
for p in report_params:
    if p == "wind_direction_deg":
        continue   # circular -- skip band export, sin/cos bands aren't meaningful in degrees
    unc[f"{p}__actual"] = truth[p].values
    unc[f"{p}__deepar_mean"] = deepar_final[p].values
    unc[f"{p}__deepar_std"] = deepar_std_df[p].values if p in deepar_std_df.columns else np.nan
    unc[f"{p}__gp_mean"] = gp_final[p].values
    unc[f"{p}__gp_std"] = gp_std_df[p].values if p in gp_std_df.columns else np.nan
unc.to_csv("uncertainty_bands.csv", index=False)
print("Saved: uncertainty_bands.csv (DeepAR Monte-Carlo std + GP posterior std per parameter)")


## 19. How to read the results

With 8 models now compared, a few patterns emerge beyond the original LSTM/XGBoost/N-BEATS/N-HiTS findings:

- **The Harmonic-Residual Hybrid should dominate `tidal_level_m`** — it directly encodes the M2/K1/O1 periods the data was generated from, and the deterministic harmonic part doesn't degrade with horizon. This validates the literature's standing recommendation (VMD/harmonic + residual ML) for tidal forecasting.
- **DLinear's strength is a sanity check**: if it ties or beats the deep models on a parameter, that parameter's structure is mostly linear/seasonal and doesn't need a complex architecture in production.
- **TSMixer's feature-mixing should help most on coupled parameters** (wind speed -> wave height, pressure -> wind) where cross-parameter information matters.
- **TiDE's covariate-aware dense encoder** is a good default when you need Transformer-tier accuracy without Transformer-tier training cost.


- **N-BEATS and N-HiTS forecast the full 120-hour horizon directly in one shot** — no
  recursive rollout, so unlike the LSTM they cannot drift from compounding one-step
  errors. This generally helps most on the smoother, more periodic parameters (tides,
  current speed, diurnal cycles) where the long-range shape matters more than
  step-to-step reactivity.
- **XGBoost** (also direct multi-horizon) remains a very strong, fast baseline — direct
  tree ensembles are hard to beat on tabular lag features for the noisier
  weather-driven parameters (air pressure, wind speed, visibility).
- **N-HiTS's multi-rate pooling** (24h/12h/1h) is designed to pay off most on series with
  clear multi-scale structure (tidal level, current speed, solar radiation); on
  parameters that are dominated by short, irregular bursts (precipitation) the coarser
  pooled stacks have less to work with.
- Compare `best_model` per row in `metrics_summary_pytorch.csv` — for production, build
  an ensemble that picks the best model per parameter, the same recommendation as the
  TensorFlow notebook.
- Open the **Streamlit dashboard** (`streamlit run app.py`) to interactively explore every
  parameter, every model, and the skill comparison.


## 20. Best model per parameter (ranked)

A simple ranked view of `metrics_df`: which single model wins each parameter, sorted by
**how decisively** it wins (its skill vs persistence), strongest result first. This is
the same `best_model` column computed in §12, just re-sorted for readability — no new
computation, just a ranking.

In [ ]:
def best_skill_value(row):
    return row[f"{row['best_model']}_skill_%"]

ranking_df = metrics_df.copy()
ranking_df["best_skill_%"] = ranking_df.apply(best_skill_value, axis=1)
ranking_df = ranking_df.sort_values("best_skill_%", ascending=False)
ranking_cols = ["parameter", "best_model", "best_skill_%"] + [f"{m}_MAE" for m in MODELS]
ranking_df = ranking_df[ranking_cols].reset_index(drop=True)
ranking_df.insert(0, "rank", ranking_df.index + 1)
ranking_df.to_csv("best_model_per_parameter.csv", index=False)
ranking_df


## 21. Two-model ensembles — can combining models beat the single best?

For every parameter, this tests **all 28 possible pairs** of the 8 models with a
**simple unweighted average** of their predictions, and checks whether any pair beats
the single best individual model.

**Why simple averaging, and not a fitted/weighted blend?** Fitting blend weights (e.g.
inverse-MAE weighting) requires a validation window *separate* from the 120-hour test
set used for scoring — we only have one held-out window here, so fitting weights against
it would leak test information into the "ensemble" and overstate its benefit. A plain
50/50 average has **zero free parameters**, so any improvement it shows is a genuine,
leakage-free signal that two models' errors partially cancel — not an artifact of tuning
against the test set. This is the practical, production-safe choice.

Wind direction is circular, so its ensemble uses a vector (sin/cos) average rather than
a naive average of degrees.

In [ ]:
from itertools import combinations

def circular_mean_deg(deg_arrays):
    rad = np.deg2rad(np.stack(deg_arrays, axis=0))
    sin_mean = np.sin(rad).mean(axis=0)
    cos_mean = np.cos(rad).mean(axis=0)
    return np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360

model_names = list(MODELS.keys())
ensemble_rows = []
ensemble_series = {}   # parameter -> best-pair ensemble forecast (for plotting)

for p in report_params:
    yt = truth[p].values
    is_circular = (p == "wind_direction_deg")
    yp_persist = np.repeat(last_obs[p], HORIZON)
    mae_p = circ_mae(yt, yp_persist) if is_circular else mean_absolute_error(yt, yp_persist)

    row = metrics_df[metrics_df["parameter"] == p].iloc[0]
    single_best_model = row["best_model"]
    single_best_mae = row[f"{single_best_model}_MAE"]

    best_pair, best_pair_mae, best_pair_series = None, np.inf, None
    for m1, m2 in combinations(model_names, 2):
        pred1, pred2 = MODELS[m1][p].values, MODELS[m2][p].values
        if is_circular:
            ens = circular_mean_deg([pred1, pred2])
            mae = circ_mae(yt, ens)
        else:
            ens = (pred1 + pred2) / 2.0
            mae = mean_absolute_error(yt, ens)
        if mae < best_pair_mae:
            best_pair, best_pair_mae, best_pair_series = (m1, m2), mae, ens

    best_pair_skill = (1 - best_pair_mae / mae_p) * 100 if mae_p > 0 else np.nan
    improvement = single_best_mae - best_pair_mae
    recommended = best_pair_mae < single_best_mae - 1e-9

    ensemble_series[p] = best_pair_series
    ensemble_rows.append({
        "parameter": p,
        "single_best_model": single_best_model,
        "single_best_MAE": round(single_best_mae, 4),
        "best_ensemble_pair": f"{best_pair[0]} + {best_pair[1]}",
        "ensemble_MAE": round(best_pair_mae, 4),
        "ensemble_skill_%": round(best_pair_skill, 1),
        "improvement_vs_single_best": round(improvement, 4),
        "recommended_ensemble": bool(recommended),
    })

ensemble_df = pd.DataFrame(ensemble_rows).sort_values("improvement_vs_single_best", ascending=False)
ensemble_df.to_csv("ensemble_recommendation.csv", index=False)
print(f"Ensembling helps on {ensemble_df['recommended_ensemble'].sum()} / {len(ensemble_df)} parameters.")
ensemble_df


## 22. Save ensemble forecasts for the dashboard

Saves the best-pair ensemble forecast for every parameter (whether or not it beat the
single best model) alongside the actual values, so the dashboard can plot "ensemble vs.
single-best vs. actual" on demand.

In [ ]:
ens_fva = pd.DataFrame({"timestamp": truth.index})
for p in report_params:
    ens_fva[f"{p}__actual"] = truth[p].values
    ens_fva[f"{p}__ensemble"] = ensemble_series[p]
ens_fva.to_csv("ensemble_forecast_vs_actual.csv", index=False)
print("Saved: best_model_per_parameter.csv, ensemble_recommendation.csv, ensemble_forecast_vs_actual.csv")


## 23. Ensemble takeaways

- Ensembling **only helps on a subset of parameters** — exactly the practical, "don't
  force it" outcome we want. Where two models' errors are correlated (both miss the same
  storm spike), averaging doesn't help and can even hurt; where their errors are
  *different in kind* (e.g. a smooth periodic model + a noisy reactive model), averaging
  cancels some of each other's mistakes.
- `recommended_ensemble = True` rows in `ensemble_recommendation.csv` are the parameters
  where a production system should serve the averaged pair instead of any single model.
- For everything else, serve the single best model from §16 — adding a second model adds
  inference cost (see the Real-Time tab) for no accuracy benefit, so it fails the
  practicality test even when it doesn't hurt accuracy.
